# S00 · Molecules → Typed Graphs + Graph Matching (RDKit → NetworkX)

<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible reaction modeling.
</div>

<div class="alert alert-block alert-success">
<b>What you will gain</b>: a robust, inspectable representation of molecules as <b>typed graphs</b>, plus the core matching concepts that power reaction modeling by graph transformation: <b>morphisms</b>, <b>isomorphism</b>, <b>automorphisms</b>, <b>subgraph isomorphism</b>, and <b>MCS</b>.  
You will also see how RDKit’s chemistry-aware matching compares to pure graph-based matching in NetworkX.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility note</b>:  
Graph matching results depend on RDKit version, attribute choices, and symmetry handling.
Always record package versions, matching predicates, and MCS settings.
Export intermediate molecular graphs (GraphML / JSON) and match mappings for inspection and
debugging.
</div>


---

## Aim of this talktorial

Reaction modeling via **graph transformation** (e.g. DPO rules) relies on two pillars:

1. **Representation** — converting molecules into graphs with chemically meaningful labels (typed graphs).
2. **Matching** — reliably identifying when two graphs (or parts of graphs) correspond, via **typed graph morphisms**:
   - isomorphisms,
   - automorphisms,
   - subgraph isomorphisms,
   - MCS-based alignments.

This talktorial establishes both pillars with minimal, transparent implementations using:

- **RDKit** as the chemical ground truth (sanitization, aromaticity, valence, MCS),
- **NetworkX** as the generic graph engine for matching, automorphism analysis, and later rewriting.

**Data example:** `data/molecules.csv`  

---

## Learning outcomes

After completing this talktorial, you will be able to:

- Convert SMILES into a **typed NetworkX molecular graph** (atoms → nodes, bonds → edges).
- Perform a **round-trip**: RDKit → NetworkX → RDKit and check what is preserved.
- State and use the formal definition of a **typed graph morphism** and its special cases:
  - homomorphism, monomorphism, isomorphism, automorphism.
- Distinguish and apply:
  - **graph isomorphism** (same structure under relabeling),
  - **automorphisms** (symmetries of one graph),
  - **subgraph isomorphism** (pattern inside host),
  - **MCS** (maximum common substructure) as a practical alignment primitive.
- Compare **RDKit morphisms** (SMARTS-based, chemistry-aware) to **NetworkX morphisms** (pure structure+attributes).
- Understand why **symmetry** and **attribute choices** strongly influence rule extraction and rule application later in SynEdu.

---

## Roadmap

0. **Setup & data**
1. **Theory: typed graphs and morphisms**
2. **RDKit → NetworkX typed graphs**
3. **Round-trip: NetworkX → RDKit and sanity checks**
4. **Isomorphism & automorphisms**
5. **Subgraph isomorphism (pattern → host)**
6. **MCS alignment with RDKit**
7. **RDKit vs NetworkX morphisms: comparison**
8. **Discussion, quiz, and references**


## 0. Setup & data

In [3]:
import rdkit
from rdkit import Chem
from rdkit.Chem import rdFMCS
import networkx as nx
import pandas as pd
from pathlib import Path

print("RDKit version:", rdkit.__version__)
print("NetworkX version:", nx.__version__)


RDKit version: 2025.03.3
NetworkX version: 3.4.2


In [5]:
DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "molecules.csv"
df = pd.read_csv(CSV_PATH)
display(df)

,smiles,name
0,CCO,ethanol
1,CCCl,chloroethane
2,CCOCC,diethyl ether
3,O,water
4,C(C(=O)O)N,glycine


# 1. Theory: Typed Graphs and Morphisms

## 1.1 Typed molecular graphs

In computational reaction modeling, we represent molecules as **typed graphs** so that “matching” respects
chemical identity (elements, charges, bond orders), not just connectivity.

A **typed graph** is a quadruple

$$
G = (V, E, \tau_V, \tau_E),
$$

where:

- **Vertices** $V$ represent **atoms**.
- **Edges** $E \subseteq \{\{u,v\}\mid u,v\in V,\ u\neq v\}$ represent **bonds** (finite, undirected, simple: no loops, no parallel edges).
- $\tau_V: V \to \mathcal{A}_V$ assigns **atom attributes** (chemical labels).
- $\tau_E: E \to \mathcal{A}_E$ assigns **bond attributes** (chemical labels).

We often write $V(G)$ and $E(G)$ for the vertex and edge sets of $G$. For a vertex $v\in V(G)$:

- neighbourhood:
  $$
  N_G(v)=\{w\in V(G)\mid vw\in E(G)\},
  $$
- degree:
  $$
  \deg_G(v)=|N_G(v)|.
  $$

### Labels (typed graphs)

“Types” are encoded as labelling maps

$$
\ell_V: V(G)\to L_V,\qquad \ell_E: E(G)\to L_E,
$$

where $L_V$ and $L_E$ are finite, non-empty label sets.
For molecular graphs we use the chemistry-specific notation:

$$
a_G: V(G)\to L_V \quad\text{(atom labels)},\qquad
b_G: E(G)\to L_E \quad\text{(bond labels)}.
$$

Let $\mathcal{G}$ denote the class of all labelled molecular graphs equipped with $(a_G,b_G)$.
In chemistry, $a_G(v)$ encodes *what atom this is* (element, charge, aromaticity, …), 
while $b_G(uv)$ encodes *what bond this is* (order, aromaticity, ring status, …).

---

## 1.2 Graph morphisms

Let $G,H\in\mathcal{G}$ be labelled molecular graphs with atom- and bond-labelling functions
$(a_G,b_G)$ and $(a_H,b_H)$.

A **(labelled) graph morphism** from $G$ to $H$ is a map

$$
\varphi:V(G)\to V(H)
$$

that preserves **chemical identity at atoms**, and preserves **bonds and their types**.
Formally, $\varphi$ must satisfy:

### (M1) Atom-label preservation
For every atom $v\in V(G)$,

$$
a_H(\varphi(v)) = a_G(v).
$$

> **Chemistry meaning**  
> $\varphi$ never maps a carbon to a nitrogen, or a neutral atom to a charged atom, if those are encoded in $a_\cdot$.

### (M2) Adjacency (bond existence) preservation
For every bond $uv\in E(G)$,

$$
\varphi(u)\varphi(v)\in E(H).
$$

> **Chemistry meaning**  
> Bonded atoms in $G$ must map to bonded atoms in $H$ (no bond can “disappear” under the map).

### (M3) Bond-label preservation
For every bond $uv\in E(G)$,

$$
b_H\!\big(\varphi(u)\varphi(v)\big) = b_G(uv).
$$

> **Chemistry meaning**  
> If $uv$ is a double bond in $G$, its image must be a double bond in $H$ (and likewise for aromaticity if included).

> **Summary**  
> A morphism $\varphi$ is a label-preserving embedding of the local chemical graph of $G$ into $H$.  
> It preserves “what atoms are” and “how they are connected”.

---

### Compatibility predicates (practical generalization)

In practice, chemoinformatics representations may differ (e.g. aromatic vs Kekulé, resonance conventions).
We therefore sometimes relax strict equality into **compatibility**:

- atom compatibility:
  $$
  a_H(\varphi(v)) \sim_V a_G(v),
  $$
- bond compatibility:
  $$
  b_H(\varphi(u)\varphi(v)) \sim_E b_G(uv),
  $$

where $\sim_V$ and $\sim_E$ encode allowed correspondences (e.g. “aromatic bond” compatible with alternating single/double under a chosen model).

> **For chemists**  
> This is where you decide whether two representations should be considered “the same chemistry”.  
> For example, strict matching distinguishes aromatic vs Kekulé; compatibility matching can treat them as equivalent.

---

### Standard special cases (matching tasks)

- **Monomorphism (injective morphism)**  
  $\varphi$ is injective (distinct atoms in $G$ map to distinct atoms in $H$).  
  $\rightarrow$ This is the formal object behind **substructure search / subgraph isomorphism**.

- **Isomorphism**  
  $\varphi$ is bijective and $\varphi^{-1}$ is also a morphism.  
  $\rightarrow$ We write $G\simeq H$ (same molecule up to relabeling).

- **Automorphism**  
  An isomorphism $\varphi:G\to G$.  
  $\rightarrow$ Encodes **molecular symmetry**, which can multiply equivalent matches and motivates deduplication.

> **Interpretation for reaction rules**  
> Applying a graph-rewrite rule begins by finding a **monomorphism** from the rule LHS into the host molecule.  
> Automorphisms (symmetry) can generate many equivalent embeddings; later SynEdu notebooks introduce symmetry-aware deduplication.

## 2. RDKit ⇄ NetworkX: Typed Molecular Graphs

In SynEdu, **RDKit** and **NetworkX** play complementary roles:

- **RDKit** is the chemical authority: sanitization, valence rules, aromaticity perception, and canonicalization.
- **NetworkX** provides an explicit, inspectable graph representation used for matching, symmetry analysis,
  and later graph rewriting.

To ensure that graph-based operations remain chemically meaningful, we require a **reversible interface**
between the two representations.

In [ ]:
from typing import Dict
import networkx as nx
from rdkit import Chem
import rdkit


In [28]:
# RDKit -> NetworkX

def mol_to_graph(mol: Chem.Mol, include_implicit_h: bool = True) -> nx.Graph:
    """
    Convert RDKit Mol -> typed NetworkX graph.

    :param mol: RDKit Mol (assumed sanitized).
    :param include_implicit_h: If True, store total H count per atom as ``total_h``.
    :returns: networkx.Graph with atom/bond labels as node/edge attributes.
    """
    G = nx.Graph()

    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        attrs: Dict[str, object] = {
            "symbol": atom.GetSymbol(),
            "formal_charge": int(atom.GetFormalCharge()),
            "aromatic": bool(atom.GetIsAromatic()),
            "chiral_tag": str(atom.GetChiralTag()),
        }
        if include_implicit_h:
            attrs["total_h"] = int(atom.GetTotalNumHs())
        G.add_node(i, **attrs)

    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        order = int(round(bond.GetBondTypeAsDouble()))
        G.add_edge(
            u, v,
            order=order,
            aromatic=bool(bond.GetIsAromatic()),
            in_ring=bool(bond.IsInRing()),
        )

    G.graph["source"] = "rdkit"
    G.graph["rdkit_version"] = rdkit.__version__
    return G


In [29]:
# NetworkX to rdkit
def graph_to_mol(G: nx.Graph, make_explicit_h: bool = False) -> Chem.Mol:
    """
    Reconstruct RDKit Mol from typed NetworkX graph (inverse of ``mol_to_graph`` up to sanitization).

    :param G: Typed molecular graph produced by ``mol_to_graph``.
    :param make_explicit_h: If True and ``total_h`` exists, add explicit H atoms (best-effort).
    :returns: Sanitized RDKit Mol.
    """
    rw = Chem.RWMol()
    nx_to_rdk: Dict[int, int] = {}

    # atoms
    for node in sorted(G.nodes()):
        n = G.nodes[node]
        atom = Chem.Atom(n.get("symbol", "C"))
        atom.SetFormalCharge(int(n.get("formal_charge", 0)))
        if n.get("aromatic", False):
            atom.SetIsAromatic(True)

        ch_tag = n.get("chiral_tag")
        if ch_tag and ch_tag != "CHI_UNSPECIFIED":
            try:
                atom.SetChiralTag(getattr(Chem.rdchem.ChiralType, ch_tag))
            except Exception:
                pass  # best-effort only

        nx_to_rdk[node] = rw.AddAtom(atom)

    # bonds
    for u, v, e in G.edges(data=True):
        order = int(e.get("order", 1))
        btype = {
            1: Chem.rdchem.BondType.SINGLE,
            2: Chem.rdchem.BondType.DOUBLE,
            3: Chem.rdchem.BondType.TRIPLE,
        }.get(order, Chem.rdchem.BondType.SINGLE)
        rw.AddBond(nx_to_rdk[u], nx_to_rdk[v], btype)

    mol = rw.GetMol()

    # optional explicit H
    if make_explicit_h:
        for node, rdk_idx in nx_to_rdk.items():
            total_h = G.nodes[node].get("total_h")
            if total_h is None:
                continue
            atom = mol.GetAtomWithIdx(rdk_idx)
            current_h = sum(1 for n in atom.GetNeighbors() if n.GetSymbol() == "H")
            for _ in range(max(int(total_h) - current_h, 0)):
                h_idx = mol.AddAtom(Chem.Atom("H"))
                mol.AddBond(rdk_idx, h_idx, Chem.rdchem.BondType.SINGLE)

    Chem.SanitizeMol(mol)
    return mol


## Exercise: Round-trip accuracy (RDKit ⇄ NetworkX)

The goal of this exercise is to verify that converting

RDKit → NetworkX → RDKit

preserves the **chemical information we care about**.

You should treat the two functions provided above as a black box.

---

### Q1 — Heavy-atom SMILES invariance

Write a function `roundtrip_smiles_equal(smiles)` that:

1. parses a SMILES string into an RDKit molecule,
2. converts it to a typed graph using `mol_to_graph`,
3. reconstructs a molecule using `graph_to_mol`,
4. compares the **canonical heavy-atom SMILES** of the original and reconstructed molecules.

The function should return `True` if the two SMILES are identical, and `False` otherwise.

---

### Q2 — Count invariants

Extend your check in **Q1** to also verify that:

- the number of **heavy atoms** is preserved,
- the number of **heavy-atom bonds** is preserved.

Return `True` only if *all* invariants are satisfied.

> Hint: use `Chem.RemoveHs(mol)` before counting atoms or bonds.

---


<details>
<summary><b>Solution:</b></summary>

### Q1–Q2: Round-trip checker (heavy SMILES + count invariants)

```python
from rdkit import Chem
from rdkit.Chem import rdmolops  # optional: useful for extra invariants

def canonical_heavy_smiles(m: Chem.Mol) -> str:
    """Return canonical SMILES after removing H (heavy-atom skeleton)."""
    return Chem.MolToSmiles(Chem.RemoveHs(m), canonical=True)

def heavy_counts(m: Chem.Mol) -> tuple[int, int]:
    """Return (n_heavy_atoms, n_heavy_bonds) after removing H."""
    mh = Chem.RemoveHs(m)
    return mh.GetNumAtoms(), mh.GetNumBonds()

def roundtrip_ok(smiles: str, verbose: bool = True) -> bool:
    """
    RDKit → NetworkX → RDKit round-trip check.

    Criteria:
    1) canonical heavy-atom SMILES preserved
    2) heavy atom count preserved
    3) heavy bond count preserved
    """
    m1 = Chem.MolFromSmiles(smiles)
    if m1 is None:
        if verbose:
            print("Parse failed:", smiles)
        return False

    G = mol_to_graph(m1, include_implicit_h=True)
    m2 = graph_to_mol(G, make_explicit_h=False)

    s1, s2 = canonical_heavy_smiles(m1), canonical_heavy_smiles(m2)
    c1, c2 = heavy_counts(m1), heavy_counts(m2)

    ok = (s1 == s2) and (c1 == c2)

    if verbose and not ok:
        print("FAIL:", smiles)
        print(" heavy SMILES:", s1, "vs", s2)
        print(" counts:", c1, "vs", c2)

    return ok
```

### Quick test (run on a small subset)

```python
n_test = min(50, len(df))
fails = []

for s in df["smiles"].head(n_test):
    if not roundtrip_ok(s, verbose=False):
        fails.append(s)

print("Checked:", n_test)
print("Failures:", len(fails))
if fails:
    print("Example failures:", fails[:5])

# Optional: inspect one failure in detail
if fails:
    _ = roundtrip_ok(fails[0], verbose=True)
```


# 3. Isomorphism

We connect the **formal morphism definitions** to concrete `networkx` matchers.

We define attribute predicates \(\Phi_V\) and \(\Phi_E\) as Python functions:
- `node_match(n1, n2)`
- `edge_match(e1, e2)`

For S01 we use strict-but-minimal compatibility:
- same `symbol`,
- same `formal_charge`,
- same `aromatic`,
- same bond `order`.


In [35]:
from rdkit import Chem
from networkx.algorithms import isomorphism as iso

pairs = {
    "benzene": ("c1ccccc1", "C1=CC=CC=C1"),
    "aniline": ("c1ccccc1N", "c1ccccc1[NH3+]"),
}

graphs = {}
for name, (sa, sb) in pairs.items():
    graphs[f"{name}_a"] = mol_to_graph(Chem.MolFromSmiles(sa))
    graphs[f"{name}_b"] = mol_to_graph(Chem.MolFromSmiles(sb))


def node_match(n1, n2):
    return n1.get("symbol") == n2.get("symbol")

def edge_match(e1, e2):
    return int(e1.get("order", 1)) == int(e2.get("order", 1))

def iso_and_count(G1, G2, nm, em):
    gm = iso.GraphMatcher(G1, G2, node_match=nm, edge_match=em)
    return gm.is_isomorphic(), sum(1 for _ in gm.isomorphisms_iter())

print("=== simple matcher (symbol + order) ===")
for name in pairs:
    G1 = graphs[f"{name}_a"]; G2 = graphs[f"{name}_b"]
    iso_flag, n_maps = iso_and_count(G1, G2, node_match, edge_match)
    print(f"{name:8} | isomorphic: {int(iso_flag):1d} | mappings: {n_maps}")


=== simple matcher (symbol + order) ===
benzene  | isomorphic: 1 | mappings: 12
aniline  | isomorphic: 1 | mappings: 2


## Exercise: Isomorphism
**Q3 — Fix the matcher**

Implement `node_match` that requires matching `symbol` **and** either `total_h` or `formal_charge` (or both). Replace the existing `node_match` with your function and re-run the demo so that:

- `benzene` still matches, and  
- `aniline` (`c1ccccc1N`) **does not** match `anilinium` (`c1ccccc1[NH3+]`).

> Hint: `mol_to_graph(..., include_implicit_h=True)` stores H as `total_h`. Use `n.get("total_h",0)` or `n.get("formal_charge",0)`.



<details> <summary><b>Solution:</b></summary>

```python
# Solution: enhanced matcher that checks symbol + (total_h OR formal_charge)
def enhanced_node_match(n1, n2):
    return (
        n1.get("symbol") == n2.get("symbol")
        and (
            int(n1.get("total_h", 0)) == int(n2.get("total_h", 0))
            or int(n1.get("formal_charge", 0)) == int(n2.get("formal_charge", 0))
        )
    )

# run the demo with the enhanced matcher (uses existing `pairs`, `graphs`, `edge_match`, `iso_and_count`)
print("=== enhanced matcher (symbol + total_h/charge) ===")
for name in pairs:
    G1 = graphs[f"{name}_a"]; G2 = graphs[f"{name}_b"]
    iso_flag, n_maps = iso_and_count(G1, G2, enhanced_node_match, edge_match)
    print(f"{name:8} | isomorphic: {int(iso_flag):1d} | mappings: {n_maps}")

# quick instructor checks
assert iso_and_count(graphs["benzene_a"], graphs["benzene_b"], enhanced_node_match, edge_match)[0]
assert not iso_and_count(graphs["aniline_a"], graphs["aniline_b"], enhanced_node_match, edge_match)[0]
```
<details>

# 4. Automorphisms & orbits

**Observation.** In the benzene example you enumerated **12 mappings** — these are the automorphisms of the benzene heavy-atom graph (the dihedral group \(D_6\), where \(|D_6| = 12\)).

**Definition.** An automorphism is a graph isomorphism from the graph to itself:

$$
f : G \longrightarrow G.
$$

The automorphism group is

$$
\mathrm{Aut}(G).
$$

The **orbit** of a vertex \(v\) is the set of images of \(v\) under all automorphisms:

$$
\mathrm{Orbit}(v)=\{\psi(v)\;|\;\psi\in\mathrm{Aut}(G)\}.
$$

**Facts.** For benzene:

$$
|\mathrm{Aut}(G)| = |D_6| = 12,
$$

and all six carbon atoms lie in a single orbit.

**Why it matters.** Symmetric hosts produce many equivalent embeddings → duplicate matches and wasted work.

**Simple remedies.**
- Deduplicate by host-atom set: use `frozenset(mapping.values())`.  
- Use orbit representatives (e.g. choose the $\min$ index per orbit).  
- Accept only a canonical mapping (WL/lexicographic tie-break).

**Practical tips.**
- Include chemical attributes (`total_h`, `formal_charge`, stereochemistry) in matchers to reduce false symmetry.  
- Pre-filter with cheap signatures (degree, label counts, WL hashes) before enumerating automorphisms.



## Exercise: Automorphisms of a Molecular Graph

### Q4 — Develop a function `enumerate_automorphisms` to enumerate all automorphisms of a graph

**Hint:** An automorphism of a graph \(G\) is an isomorphism from \(G\) to itself.

Equivalently, the automorphism group satisfies

$$
\mathrm{Aut}(G) \subseteq \mathrm{Iso}(G, G).
$$



<details> <summary><b>Solution:</b></summary>

```python
def enumerate_automorphisms(G: nx.Graph):
    GM_self = iso.GraphMatcher(G, G, node_match=node_match, edge_match=edge_match)
    return list(GM_self.isomorphisms_iter())
```

We can now analyse the symmetry of a molecular graph by computing the
**orbits induced by its automorphism group**.

Under the natural action of the automorphism group on the vertex set,
two vertices belong to the same orbit if there exists an automorphism
mapping one to the other.

$$
\text{For } u, v \in V(G), \quad
u \sim v
\;\Longleftrightarrow\;
\exists\, \varphi \in \mathrm{Aut}(G)
\text{ such that }
\varphi(u) = v .
$$

Each orbit therefore represents a set of **symmetry-equivalent atoms**.


In [44]:
import networkx as nx
from typing import Dict, Iterable, List, Set


def compute_orbits_from_automorphisms(
    G: nx.Graph,
    automorphisms: Iterable[Dict] | None = None,
) -> List[Set]:
    """
    Compute vertex orbits induced by the automorphism group of a graph.

    Given the automorphism group Aut(G) acting on V(G), two vertices
    u, v ∈ V(G) belong to the same orbit if there exists an automorphism
    φ ∈ Aut(G) such that φ(u) = v.

    This function computes the orbits by collapsing vertices connected
    by automorphism mappings using a union–find (disjoint-set) structure.

    Parameters
    ----------
    G : nx.Graph
        Input graph.
    automorphisms : iterable of dict, optional
        Precomputed automorphisms φ : V(G) → V(G).
        If None, they are computed internally.

    Returns
    -------
    List[Set]
        List of vertex orbits. Each orbit is a set of nodes.
        Ordering is deterministic (sorted by smallest element).
    """
    if automorphisms is None:
        automorphisms = enumerate_automorphisms(G)

    # --- Disjoint-set (union–find) structure ---
    parent: Dict = {v: v for v in G.nodes()}

    def find(x):
        """Find representative with path compression."""
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(a, b):
        """Union sets containing a and b."""
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # --- Apply group action ---
    for auto in automorphisms:
        for v, fv in auto.items():
            union(v, fv)

    # --- Collect orbits ---
    orbits: Dict = {}
    for v in G.nodes():
        r = find(v)
        orbits.setdefault(r, set()).add(v)

    # deterministic ordering (useful for teaching & testing)
    return sorted(orbits.values(), key=lambda s: min(s))

from rdkit import Chem

benzene = Chem.MolFromSmiles("c1ccccc1")
G_bz = mol_to_graph(benzene)

autos = enumerate_automorphisms(G_bz)
orbits = compute_orbits_from_automorphisms(G_bz, autos)

print("Number of automorphisms (benzene):", len(autos))
print("Orbits:", orbits)


Number of automorphisms (benzene): 12
Orbits: [{0, 1, 2, 3, 4, 5}]


# 5. Subgraph isomorphism (pattern → host)

In rule-based reaction modeling, we repeatedly solve the **pattern-in-host** query:

$$
\text{Does a pattern graph } P \text{ occur inside a host graph } G?
\quad \text{If yes, what are the embeddings?}
$$

Formally, a **subgraph isomorphism** is an **injective, label-preserving graph morphism**

$$
f : V(P) \hookrightarrow V(G)
$$

such that:

- **Atom (node) labels are preserved**
  $$a_G(f(v)) = a_P(v)\quad \forall v \in V(P)$$

- **Bond existence and bond types are preserved**
  $$uv \in E(P)\ \Rightarrow\ f(u)f(v) \in E(G), \quad b_G(f(u)f(v)) = b_P(uv)$$

Intuitively, \(f\) is a **typed monomorphism**: it embeds the pattern into the host
without collisions (injective), while respecting chemical identity (types).

### Practical note: many matches due to symmetry
Even when the chemical occurrence is “the same”, symmetric graphs can admit many
equally valid embeddings:

- **Host symmetry** (automorphisms of \(G\)) produces multiple placements.
- **Pattern symmetry** (automorphisms of \(P\)) produces multiple equivalent mappings.
- If both are symmetric, matches can multiply combinatorially.

NetworkX exposes this via:

- `GraphMatcher.subgraph_isomorphisms_iter()` — enumerates all injective embeddings
  that satisfy `node_match` and `edge_match`.

For downstream tasks (reaction center extraction, rule application, deduplication),
we often need to **post-process** these matches to remove symmetry-equivalent
embeddings, typically by orbit-based canonicalization or choosing a canonical
representative embedding.


In [58]:
from __future__ import annotations

from collections import Counter
from typing import Dict, List, Iterable, Tuple
import networkx as nx
from networkx.algorithms import isomorphism as iso
from rdkit import Chem


def nx_subgraph_matches(
    host_G: nx.Graph,
    pattern_G: nx.Graph,
    *,
    invert: bool = True,
) -> List[Dict]:
    """
    Enumerate subgraph isomorphisms of `pattern_G` inside `host_G`.

    Notes
    -----
    NetworkX's GraphMatcher(host, pattern).subgraph_isomorphisms_iter()
    yields mappings of the form:  host_node -> pattern_node  (G1 -> G2).

    If you want the more intuitive direction (pattern -> host), set `invert=True`.
    """
    GM = iso.GraphMatcher(
        host_G,
        pattern_G,
        node_match=node_match,
        edge_match=edge_match,
    )

    out: List[Dict] = []
    for m_host_to_pat in GM.subgraph_isomorphisms_iter():
        if invert:
            out.append({p: h for h, p in m_host_to_pat.items()})
        else:
            out.append(m_host_to_pat)
    return out


# --- Symmetry-heavy example: benzene pattern in naphthalene host ---
# Pattern is highly symmetric (|Aut|=12), and the host contains two benzene rings.
# => many raw embeddings (symmetry variants)

pattern_mol = Chem.MolFromSmiles("c1ccccc1")          # benzene (symmetric)
host_mol    = Chem.MolFromSmiles("c1ccc2ccccc2c1")    # naphthalene (symmetric)

pattern_G = mol_to_graph(pattern_mol)
host_G    = mol_to_graph(host_mol)

matches = nx_subgraph_matches(host_G, pattern_G, invert=True)  # pattern_idx -> host_idx

print("Raw subgraph isomorphisms (pattern -> host):", len(matches))


Raw subgraph isomorphisms (pattern -> host): 24


## Deduplication of subgraph embeddings

Raw subgraph matches often contain many symmetry-equivalent embeddings.  
We present a minimal, renderer-friendly description of the **host-image** deduplication strategy.

Let \(m : V(P)\to V(G)\) be a pattern→host mapping. Define the **image** of \(m\) as the set of host nodes touched by the embedding:

$$
\mathrm{img}(m)\;=\;\{\,m(v)\;|\;v\in V(P)\,\}\subseteq V(G).
$$

We deduplicate embeddings by grouping all mappings that share the same image.  
Operationally we use the sorted tuple of `img(m)` as a canonical key:

$$
\text{key}(m) \;=\; \text{tuple}(\mathrm{sorted}(\mathrm{img}(m))).
$$

This collapses symmetry variants that differ only by a permutation of pattern nodes (i.e., different bijections from the same image) and yields the distinct *placements* of the pattern inside the host.


In [69]:
from collections import defaultdict
from typing import Dict, Iterable, List, Mapping, Tuple, Union, Optional


def _build_node_to_rep(
    orbits: Union[Iterable[Iterable[int]], Mapping[int, int], None]
) -> Dict[int, int]:
    """Convert `orbits` into a mapping node -> representative."""
    if orbits is None:
        return {}
    if isinstance(orbits, Mapping):
        return dict(orbits)
    node_to_rep: Dict[int, int] = {}
    for orbit in orbits:
        orbit = set(orbit)
        if not orbit:
            continue
        rep = min(orbit)
        for v in orbit:
            node_to_rep[v] = rep
    return node_to_rep


def dedup_by_host_image_with_orbits(
    matches: Iterable[Dict[int, int]],
    *,
    orbits: Union[Iterable[Iterable[int]], Mapping[int, int], None] = None,
    pattern_node_order: Optional[Iterable[int]] = None,
    return_groups: bool = False,
) -> Union[List[Dict[int, int]], Dict[Tuple[int, ...], List[Dict[int, int]]]]:
    """
    Deduplicate pattern->host mappings by canonical host-image (with optional orbit reps).

    Parameters
    ----------
    matches : iterable of dict
        Each mapping is pattern_node -> host_node.
    orbits : iterable of iterables OR mapping node->rep OR None
        If iterable-of-iterables, representative for an orbit is min(orbit).
        If mapping, it should be node -> representative.
        If None, no orbit canonicalization is applied.
    pattern_node_order : iterable of pattern node ids, optional
        Order used to form the per-mapping tuple for selecting the representative.
        If None, inferred deterministically from the first mapping (sorted keys).
    return_groups : bool (default False)
        If False (default) return a list of representative mappings (one per canonical class).
        If True return the full dict: canonical_key -> list[mappings].

    Returns
    -------
    List[dict] or Dict[tuple, list]
        See `return_groups` description.
    """
    matches = list(matches)  # materialize to allow multiple passes
    if not matches:
        return {} if return_groups else []

    node_to_rep = _build_node_to_rep(orbits)

    # infer or validate pattern node order
    if pattern_node_order is None:
        # Use sorted keys of the first mapping (deterministic)
        pattern_node_order = tuple(sorted(matches[0].keys()))
    else:
        pattern_node_order = tuple(pattern_node_order)

    # bucket by canonical key (sorted canonicalized host nodes)
    buckets: Dict[Tuple[int, ...], List[Dict[int, int]]] = defaultdict(list)
    for m in matches:
        canonical_nodes = (node_to_rep.get(h, h) for h in m.values())
        key = tuple(sorted(canonical_nodes))
        buckets[key].append(m)

    # if user requested full groups, return them (deterministic ordering by key)
    if return_groups:
        return {k: buckets[k] for k in sorted(buckets.keys())}

    # otherwise pick a deterministic representative per bucket:
    # representative = mapping with smallest tuple (m[p] for p in pattern_node_order)
    representatives: List[Dict[int, int]] = []
    for key in sorted(buckets.keys()):
        group = buckets[key]
        # compute lexicographic tuple for each mapping according to pattern_node_order
        def ordering_tuple(m: Dict[int, int]) -> Tuple[int, ...]:
            return tuple(m[p] for p in pattern_node_order)
        rep = min(group, key=ordering_tuple)
        representatives.append(rep)

    return representatives


In [67]:
# 1. compute host orbits (optional — useful if you later want to collapse host symmetry)
host_autos = enumerate_automorphisms(host_G)                       # φ : host_node -> host_node
host_orbits = compute_orbits_from_automorphisms(host_G, host_autos)  # list of sets
print("Host orbits:", host_orbits)
print()

# 2. dedup by host-image (default: returns representatives list)
representatives = dedup_by_host_image_with_orbits(matches)  # list of rep mappings
print("Number of representatives:", len(representatives))
for i, rep in enumerate(representatives, start=1):
    print(f" representative #{i}:", rep)
print()

# 3. (optional) show full groups and multiplicities to verify symmetry inflation
groups = dedup_by_host_image_with_orbits(matches, return_groups=True)  # dict: key -> list[mappings]
print("Distinct placements by host-image (groups):", len(groups))
for key, group in groups.items():
    print(" placement key:", key, " multiplicity:", len(group))
    print("  example mapping:", group[0])
print()

# 4. (optional) collapse placements modulo host automorphisms (use host_orbits)
groups_mod_host = dedup_by_host_image_with_orbits(matches, orbits=host_orbits, return_groups=True)
print("Unique classes modulo host automorphisms:", len(groups_mod_host))
for key, group in groups_mod_host.items():
    print(" canonical_key (orbits reps):", key, " multiplicity:", len(group))
    print("  example mapping:", group[0])


Host orbits: [{0, 1, 5, 6}, {9, 2, 4, 7}, {8, 3}]

Number of representatives: 2
 representative #1: {0: 0, 1: 1, 2: 2, 3: 3, 4: 8, 5: 9}
 representative #2: {0: 3, 1: 4, 2: 5, 3: 6, 4: 7, 5: 8}

Distinct placements by host-image (groups): 2
 placement key: (0, 1, 2, 3, 8, 9)  multiplicity: 12
  example mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 8, 5: 9}
 placement key: (3, 4, 5, 6, 7, 8)  multiplicity: 12
  example mapping: {0: 3, 1: 8, 2: 7, 3: 6, 4: 5, 5: 4}

Unique classes modulo host automorphisms: 1
 canonical_key (orbits reps): (0, 0, 2, 2, 3, 3)  multiplicity: 24
  example mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 8, 5: 9}


## Exercise: Deduplicate modulo pattern automorphisms

### Q5 — Implement `dedup_by_pattern_image_with_orbits`

**Goal.**  
Implement a function `dedup_by_pattern_image_with_orbits(matches, pattern_autos, ...)` that groups/filters pattern→host mappings **modulo pattern automorphisms**.

Intuitively, two mappings \(m_1,m_2: P \to H\) are equivalent if there exists a pattern automorphism \(\varphi \in \mathrm{Aut}(P)\) such that
$$
m_2 \;=\; m_1 \circ \varphi .
$$
Equivalently, \(m_1\) and \(m_2\) differ only by a permutation of the pattern nodes.


<details> <summary><b>Solution:</b></summary>

```python
from collections import defaultdict
from typing import Dict, Iterable, List, Tuple, Optional, Union

def dedup_by_pattern_image_with_orbits(
    matches: Iterable[Dict[int,int]],
    *,
    pattern_autos: Iterable[Dict[int,int]],
    pattern_node_order: Optional[Iterable[int]] = None,
    return_groups: bool = False,
) -> Union[List[Dict[int,int]], Dict[Tuple[int,...], List[Dict[int,int]]]]:
    matches = list(matches)
    if not matches:
        return {} if return_groups else []

    # deterministic pattern node order
    if pattern_node_order is None:
        pattern_node_order = tuple(sorted(matches[0].keys()))
    else:
        pattern_node_order = tuple(pattern_node_order)

    # pre-list autos for repeated use
    autos = list(pattern_autos)

    # helper: canonical tuple for mapping m under all pattern automorphisms
    def canonical_tuple_for_mapping(m: Dict[int,int]) -> Tuple[int, ...]:
        best = None
        for phi in autos:
            tup = tuple(m[phi[p]] for p in pattern_node_order)
            if best is None or tup < best:
                best = tup
        return best

    # bucket by canonical tuple
    buckets: Dict[Tuple[int,...], List[Dict[int,int]]] = defaultdict(list)
    for m in matches:
        key = canonical_tuple_for_mapping(m)
        buckets[key].append(m)

    # deterministic ordering of keys
    ordered_keys = sorted(buckets.keys())

    if return_groups:
        return {k: buckets[k] for k in ordered_keys}

    # select deterministic representative per bucket:
    # pick mapping with smallest tuple according to pattern_node_order
    def ordering_tuple(m: Dict[int,int]) -> Tuple[int, ...]:
        return tuple(m[p] for p in pattern_node_order)

    representatives: List[Dict[int,int]] = []
    for k in ordered_keys:
        group = buckets[k]
        rep = min(group, key=ordering_tuple)
        representatives.append(rep)

    return representatives

```

# 6. MCS (Maximum Common Substructure) — RDKit + NetworkX views

The **Maximum Common Substructure (MCS)** problem asks for the largest subgraph that two molecular graphs share.
It is a core *alignment primitive* used in similarity, substructure transfer, and as a common starting point for **atom mapping**.

---

## 6.1 Formal definition (typed molecular graphs)

Let \(G=(V_G,E_G,a_G,b_G)\) and \(H=(V_H,E_H,a_H,b_H)\) be **typed** molecular graphs with
atom labels \(a(\cdot)\) (e.g., element, charge, aromatic) and bond labels \(b(\cdot)\) (e.g., order, aromatic).

A graph \(S\) is a **common subgraph** of \(G\) and \(H\) if there exist **injective typed morphisms**
(subgraph embeddings)

$$
f: S \hookrightarrow G, \qquad g: S \hookrightarrow H
$$

such that labels and bonds are preserved (same predicates as subgraph isomorphism).

An **MCS** is any common subgraph \(S^\*\) that maximizes a size objective:

$$
S^\* \in \arg\max_{S}
\Big( \, |V(S)| \;\; \text{or} \;\; w_V|V(S)| + w_E|E(S)| \, \Big)
\quad \text{s.t. } S \hookrightarrow G \text{ and } S \hookrightarrow H.
$$

**Notes.**
- The MCS need not be unique (multiple maximum solutions may exist).
- Constraints (ring-only matching, atom types, bond types, chirality) change the feasible set and thus the MCS.

---

## 6.2 RDKit view: MCS as a SMARTS pattern + match lists

RDKit provides a practical MCS solver:

- `rdFMCS.FindMCS([mol1, mol2], ...)` returns an object whose `smartsString`
  encodes a **query substructure** \(Q\) (SMARTS) intended to represent a largest shared substructure under constraints.

We then compute embeddings of the SMARTS query into each molecule:

$$
\mathrm{Match}_G(Q) = \{\, f_i : V(Q)\hookrightarrow V(G)\,\}, \qquad
\mathrm{Match}_H(Q) = \{\, g_j : V(Q)\hookrightarrow V(H)\,\}.
$$

In practice:
- `mol.GetSubstructMatches(query)` returns many embeddings (symmetry variants).
- When multiple matchings exist, choosing a canonical representative (or a best-scoring one) is a separate step.

**Interpretation.**  
RDKit’s MCS output gives you:
1) a *candidate* maximum common substructure (as a query), and  
2) potentially many embeddings in each molecule.

---

## 6.3 NetworkX view: MCS as a maximum common *subgraph isomorphism*

When we convert molecules to NetworkX graphs, MCS corresponds to finding a largest typed graph \(S\)
that is simultaneously subgraph-isomorphic to both graphs.

Conceptually:

$$
S \subseteq G,\; S \subseteq H
\quad\Longleftrightarrow\quad
\exists\, f: S\hookrightarrow G,\; \exists\, g: S\hookrightarrow H.
$$

In the NX world, this is typically attacked by:
- searching over candidate node/bond subsets, or
- using MCS heuristics (e.g., expand from seeds; branch-and-bound; constraint propagation),
because exact MCS is NP-hard.

**Why still use NX here?**
- You can enforce *your* exact chemical typing predicates (`node_match`, `edge_match`).
- You can integrate symmetry handling (automorphism orbits) and custom constraints.
- You can expose intermediate states for teaching (what gets pruned, what expands).

---

## 6.4 Symmetry and non-uniqueness (important for mapping)

Both RDKit and NX perspectives share the same caveats:

- **Multiple maximum solutions:** \(|V(S^\*)|\) may be achieved by several distinct subgraphs.
- **Many embeddings:** even for one fixed \(S^\*\), symmetry can produce many matches.
- This is why MCS is a *good starting point* for atom mapping, but not the full solution:
  you still need a tie-breaker / scoring rule to pick a consistent alignment.

---

## 6.5 Practical pipeline (RDKit ↔ NX)

A common didactic workflow is:

1. **RDKit MCS**
   - compute a SMARTS query \(Q\) using `rdFMCS.FindMCS`.
2. **RDKit embeddings**
   - enumerate `GetSubstructMatches(Q)` for each molecule.
3. **NX analysis**
   - convert selected embeddings to NX node correspondences,
   - optionally deduplicate symmetry-equivalent matches using orbits,
   - use the resulting correspondence as a seed for atom mapping or reaction-center extraction.

This makes MCS an excellent bridge between *chemistry-native toolkits* (RDKit) and *graph-theoretic control* (NetworkX).


In [ ]:
def rdkit_mcs_smarts(mols, timeout=10, ringMatchesRingOnly=True):
    res = rdFMCS.FindMCS(
        mols,
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        bondCompare=rdFMCS.BondCompare.CompareOrders,
        ringMatchesRingOnly=ringMatchesRingOnly,
        timeout=timeout,
    )
    return res.smartsString

m1 = Chem.MolFromSmiles("Oc1ccccc1")   # phenol
m2 = Chem.MolFromSmiles("Cc1ccccc1")   # toluene

mcs_smarts = rdkit_mcs_smarts([m1, m2])
print("MCS SMARTS:", mcs_smarts)

mcs_mol = Chem.MolFromSmarts(mcs_smarts)
print("MCS matches in m1:", m1.GetSubstructMatches(mcs_mol))
print("MCS matches in m2:", m2.GetSubstructMatches(mcs_mol))


# 7. RDKit vs NetworkX morphisms: comparison

- **RDKit substructure matching** is chemistry-aware and supports internal symmetry handling via `uniquify=True`.
- **NetworkX subgraph isomorphism** is pure typed-graph matching and often returns all symmetric variants unless you dedupe.

We compare match counts and host-atom sets for the same (host, pattern) pair.


In [ ]:
def rdkit_substruct_matches(host: Chem.Mol, pattern: Chem.Mol, uniquify: bool = True):
    return host.GetSubstructMatches(pattern, uniquify=uniquify)

def compare_rdkit_vs_nx(host_smiles: str, pattern_smiles: str):
    host_m = Chem.MolFromSmiles(host_smiles)
    pattern_m = Chem.MolFromSmiles(pattern_smiles)

    rd_matches = rdkit_substruct_matches(host_m, pattern_m, uniquify=True)

    host_G = mol_to_typed_nx(host_m)
    pattern_G = mol_to_typed_nx(pattern_m)
    nx_matches = nx_subgraph_matches(host_G, pattern_G)

    rd_hostsets = {tuple(sorted(t)) for t in rd_matches}
    nx_hostsets = {tuple(sorted(m.values())) for m in nx_matches}

    print("Host:", host_smiles)
    print("Pattern:", pattern_smiles)
    print("RDKit matches (uniquify=True):", len(rd_matches))
    print("NetworkX matches (raw):", len(nx_matches))
    print("NetworkX unique hostsets:", len(nx_hostsets))
    print("RDKit hostsets:", rd_hostsets)
    print("NX hostsets:", nx_hostsets)
    print("NX extra (not in RDKit hostsets):", nx_hostsets - rd_hostsets)
    print("RDKit-only (if any):", rd_hostsets - nx_hostsets)

compare_rdkit_vs_nx("Oc1ccccc1", "c1ccccc1")


# 8. Discussion (what to remember)

- A **typed graph morphism** formalizes structure- and attribute-preserving maps between graphs.
- Our **typed molecular graphs** use a minimal attribute schema (`symbol`, `formal_charge`, `aromatic`, `order`) to define what “same” means.
- **Round-trip conversion** (RDKit → NetworkX → RDKit) is valuable for debugging and peer review; we preserve heavy-atom topology, but exact RDKit internal state may differ.
- **Automorphisms** describe symmetries; they inflate match enumeration. Deduplicate (e.g. by host-atom set) to prevent combinatorial explosion.
- **Subgraph isomorphism** is the core operation for rule application later in SynEdu.
- **MCS** is a chemistry-aware alignment primitive, but it is heuristic and sometimes non-unique; always log settings and timeouts.
- **RDKit vs NetworkX**:
  - RDKit: chemistry-aware SMARTS matching, built-in `uniquify`.
  - NetworkX: full control over attributes and morphism semantics; you manage deduplication and interpretation.


# 9. Quiz

1. Formally define a **typed graph morphism** \(f: G \to H\) in one or two sentences.  
2. What additional property do we need for a morphism to be an **isomorphism**?  
3. What is an **automorphism**, and why does it matter for subgraph matching in symmetric molecules?  
4. Explain how you would deduplicate subgraph matches using **host-atom sets**.  
5. Give one practical advantage of using RDKit for substructure matching and one advantage of using NetworkX.


# 10. References and further reading

- RDKit documentation: https://www.rdkit.org/docs/  
- RDKit Book: https://www.rdkit.org/docs/Book.html  
- NetworkX documentation: https://networkx.org/documentation/stable/  
- NetworkX isomorphism: https://networkx.org/documentation/stable/reference/algorithms/isomorphism.html  
- RDKit MCS (rdFMCS): https://www.rdkit.org/docs/source/rdkit.Chem.rdFMCS.html  
